# Python程序设计（项目实战） · 工单P3《产线KPI仪表盘与自动日报》
**第5周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 数据钻取：点击折线点弹出该时段明细表（st.data_frame联动）；示范讲解 |
| **B 组长** | 标准达成 | 联动流畅+Plotly交互 |
| **C 操作员** | 基础完成 | 5KPI可用+时间筛选 |
| **D 学徒** | 跟做保底 | 跟做示例跑通任务1，程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

> **本教师版已补全 5 个任务空位（对应学生版 main() 的 5 处占位）+ 四段式日报**，
> 并在无 streamlit 环境下自动降级为 headless：同一套函数照常算 KPI、建 Plotly 图、落盘 HTML。


## 一、课前资讯（Predict·预测）
> **预习快问**：Streamlit每次交互后脚本如何重跑？缓存用哪个装饰器？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——把 starter 的 5 处待填空位补全（本教师版已全部给出完成态）；
5. **Make 创造**——独立完成工单任务。


In [3]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/03-Python程序设计-项目实战


## 三、任务要求（Make）
**实施步骤**
1. 跑通starter最小仪表盘
2. 5个KPI: 趋势/良率/状态灯/告警/对标
3. 侧栏筛选(日期+设备)+cache_data

**验收标准**（工单《P3》「验收标准」四行原文）：`及格(D)`＝`5 KPI可用+时间筛选；日报四部分完整+PDF正确` ｜ `良好(C)`＝`联动筛选流畅；日报生成<2分钟` ｜ `优秀(B)`＝`≥7指标+钻取交互；异常自动识别；美观度≥4/5` ｜ `超优(A)`＝`优秀+定时自动生成(mailto/计划任务)+移动端适配`
> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P3》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 ≥7指标+钻取交互；异常自动识别；美观度≥4/5）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。


### Starter 1：`coding: utf-8 导入块 + 离线降级`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [6]:
# -*- coding: utf-8 -*-
"""P3 产线KPI仪表盘+自动日报 —— 框架(Streamlit)
运行: streamlit run W05_dashboard.py
数据: 自动生成模拟产线数据(或替换为P2.1的clean.csv)
教师版完成态: 顶部加"离线降级"分支——装了 streamlit 走真控件渲染(课堂主路径),
             没装也照常把同一套KPI算出来打印/落盘, notebook 不依赖浏览器即可验收。
"""
import datetime
import functools
import os
import time

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:                                  # 课堂/生产环境: 真正的 Streamlit(推荐路径)
    import streamlit as st
    HAVE_ST = True
except ImportError:                   # 离线降级: 无 streamlit 时仍可跑通同一套KPI计算并打印
    HAVE_ST = False

    class _HeadlessContainer:
        """把 st.* 的"渲染意图"打印出来: 取值/分支/异常与真机一致, 无浏览器也能验收KPI"""

        def __init__(self, name="page"):
            self._name = name

        def _p(self, *args):
            print(f"[st.{self._name}]", *args)

        def set_page_config(self, page_title="", page_icon="", layout="wide", **kw):
            self._p(f"页面配置 {page_icon}{page_title} layout={layout}")

        def title(self, s, **kw): self._p("H1:", s)
        def header(self, s, **kw): self._p("H2:", s)
        def subheader(self, s, **kw): self._p("H3:", s)
        def caption(self, s, **kw): self._p("注:", s)
        def markdown(self, s, **kw): self._p("MD:", str(s).replace("\n", " ")[:90])
        def write(self, *objs, **kw): self._p("write:", *(str(o)[:100] for o in objs))

        def metric(self, label, value, delta=None, **kw):
            self._p(f"KPI卡 {label} = {value}" + (f"  Δ{delta}" if delta else ""))

        def success(self, s, **kw): self._p("🟢", s)
        def warning(self, s, **kw): self._p("🟡", s)
        def error(self, s, **kw): self._p("🔴", s)
        def info(self, s, **kw): self._p("ℹ️ ", s)

        def progress(self, value, text=None, **kw):
            v = min(max(float(value), 0.0), 1.0)          # 真机同样只吃 0~1
            n = int(round(v * 20))
            self._p(f"进度条 [{'█' * n}{'░' * (20 - n)}] {v:.1%}" + (f" {text}" if text else ""))

        def dataframe(self, frame, **kw):
            self._p(f"表格 {frame.shape[0]}行×{frame.shape[1]}列")
            print(frame.head(3).to_string(index=False))

        def plotly_chart(self, fig, **kw):
            ttl = fig.layout.title.text                        # gauge 的标题在 trace 上, 不在 layout
            if not ttl:
                ttl = getattr(getattr(fig.data[0], "title", None), "text", "") or "(无标题)"
            pts = sum(len(tr.y) for tr in fig.data
                      if getattr(tr, "y", None) is not None and hasattr(tr.y, "__len__"))
            self._p(f"Plotly图 {ttl!r}: {len(fig.data)}条轨迹 / {pts}个数据点")

        def columns(self, spec, **kw):
            n = len(spec) if isinstance(spec, (list, tuple)) else int(spec)
            return [_HeadlessContainer(f"col{i + 1}") for i in range(n)]

        def date_input(self, label, value=None, min_value=None, max_value=None, **kw):
            self._p(f"日期控件 {label} = {value}")
            return value                                  # 无交互 → 返回默认值(全区间)

        def multiselect(self, label, options, default=None, **kw):
            out = list(default) if default is not None else list(options)
            self._p(f"多选控件 {label} = {len(out)}项")
            return out                                    # 无交互 → 返回默认(全选)

        def expander(self, label, **kw): return _HeadlessContainer(f"expander:{label}")
        def tabs(self, labels, **kw): return [_HeadlessContainer(f"tab:{t}") for t in labels]

        def cache_data(self, func=None, **kw):
            """缓存降级实现: 同参数直接取上次结果(真机由 streamlit 托管缓存)"""
            def deco(f):
                memo = {}

                @functools.wraps(f)
                def wrap(*a, **k):
                    if a not in memo:
                        memo[a] = f(*a, **k)
                    return memo[a]
                wrap.clear = lambda: memo.clear()
                return wrap
            return deco(func) if callable(func) else deco

    st = _HeadlessContainer("page")
    st.sidebar = _HeadlessContainer("sidebar")

# ---- KPI 口径常量(对标的分母/阈值; 课堂按车间月度指标讲决策依据) ----
TARGET_YIELD = 0.95                       # 良率目标: 仪表盘阈值线
TARGET_OUTPUT = 105_000                   # 产量目标: 效率对标分母(定挑战值→演示"未达成"分支)
ONLINE_WINDOW = pd.Timedelta("2h")        # 最近2小时内有上报 → 判定"在线"
TEMP_LIMIT, VIB_LIMIT = 90.0, 4.5         # 告警阈值: 温度℃ / 振动mm/s
FONT_PATH = "fonts/ZH_Font.ttf"           # 02包W10中文字体方案(PDF用)
REPORT_PATH = "daily_report.html"

rng = np.random.RandomState(2026)


### Starter 2：`@st.cache_data → load_data()`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [8]:
# 教师版完成态(数据层): @st.cache_data 让 Streamlit 每次整页重跑都不重算(工单要点)
@st.cache_data
def load_data():
    """数据加载: 默认合成 2000 行半小时级产线数据(与 P2.1 产出同构)。
    可选改造(工单要求数据源=P2.1): 换成
        df = pd.read_csv("clean.csv", parse_dates=["ts"])
        并补齐 output/ok_qty/vib 三列后复用同一下游口径即可。"""
    n = 2000
    ts = pd.date_range("2026-04-01", periods=n, freq="30min")
    output = rng.poisson(50, n)
    df = pd.DataFrame({
        "ts": ts,
        "device_id": rng.choice(["DEV-01", "DEV-02", "DEV-03", "DEV-04"], n),
        "output": output,
    })
    df["ok_qty"] = (df.output * rng.uniform(0.88, 0.99, n)).astype(int)
    df["yield"] = df.ok_qty / df.output        # yield是Python关键字 → 只能 df["yield"] 取列
    df["temp"] = rng.uniform(45, 95, n)
    df["vib"] = rng.uniform(1, 6, n)
    # ---- 教师版补充: 日报"班次对比/缺陷统计"所需派生列(口径见工单P3.2) ----
    df["shift"] = pd.cut(df.ts.dt.hour, bins=[-1, 7, 15, 23],
                         labels=["夜班(0-8)", "白班(8-16)", "晚班(16-24)"])
    df["day"] = df.ts.dt.normalize()
    df["bad_qty"] = df.output - df.ok_qty
    return df


### 💡 分层提示（先自己试5分钟再看）
**思路提示**（一条对应学生版一个空位，按顺序做）
> - **任务1 侧栏筛选**：`st.sidebar.date_input("日期范围", value=(起,止))`——传**元组**才是区间；`st.sidebar.multiselect("设备", 全部设备, default=全部设备)`；两个布尔掩码用 `&` 连接后切片 `df[m1 & m2]`（`&` 优先级高，每个条件必须加括号），日期比较先降到 `df.ts.dt.date.between(d0, d1)`
> - **任务2 四个KPI指标卡**：总产量 `int(df.output.sum())`；平均良率 `df["yield"].mean()`（`yield` 是关键字，只能方括号取列）；设备在线数=最近2小时内有上报的 `df.loc[df.ts >= last - pd.Timedelta("2h"), "device_id"].nunique()`；今日告警数=`len(df[(df.temp>90)&(df.vib不满足时用|)])` 即温度>90℃ 或 振动>4.5mm/s；各配 `st.metric(名, 值, delta=环比)`，告警数这种"越少越好"的指标要 `delta_color="inverse"`
> - **任务3 产量趋势**：先按时间聚合 `t=df.groupby("ts", as_index=False).output.sum()`，再 `fig=px.line(t, x="ts", y="output", markers=True)` → `st.plotly_chart(fig, use_container_width=True)`（⚠ 只能传 plotly 图对象，matplotlib 对象塞进去会报错）
> - **任务4 良率仪表+状态灯+告警表**：仪表用 `go.Figure(go.Indicator(mode="gauge+number", value=良率*100, gauge={"axis":{"range":[0,100]}, "threshold":{"value":95}}))`；每台设备按告警条数选 `st.success`(0条)/`st.warning`(1~2条)/`st.error`(≥3条) 做红绿灯；明细 `st.dataframe(告警表)`
> - **任务5 效率对标**：`st.progress(min(实际/目标, 1.0), text=f"{比率:.1%}")`——进度条只吃 **0~1**，超目标不截断会 `ValueError`；良率对标同理用 `实际良率/目标良率`
> - **避坑(教师演示点)**：`pd.cut` 出来的分类列直接给 `px.bar(color=...)`，若某类别在筛选后没有数据 → `KeyError`，先 `.astype(str)` 或 `cat.remove_unused_categories()`
> - **日报四段式（`build_report()`）**：概览=5个KPI数值；分析=`groupby("shift")`/`groupby("device_id")` 的 `sum` + 良率列后 `.to_html()`；异常=`temp>90 或 vib>4.5` 的记录 + 按原因映射处置建议；附表=`df.tail(50).to_html()`；最后 `open(...,"w",encoding="utf-8")` 落盘

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位完成态 cell 里的「任务n」标记行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改阈值/换 `px.bar` 与 `px.area` 对比效果、给告警表加钻取联动，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`5 KPI可用+时间筛选；日报四部分完整+PDF正确` ｜ `良好(C)`＝`联动筛选流畅；日报生成<2分钟` ｜ `优秀(B)`＝`≥7指标+钻取交互；异常自动识别；美观度≥4/5` ｜ `超优(A)`＝`优秀+定时自动生成(mailto/计划任务)+移动端适配`


### Starter 3：`alarm_frame()/kpis()/main()`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [11]:
# ================= 教师版补充: KPI 统一口径(看板与日报共用, 防"两个数字对不上") =================
def alarm_frame(df):
    """告警明细口径: 温度>90℃ 或 振动>4.5mm/s → 附"告警原因+处置建议"(工单P3.2 异常识别)"""
    a = df.loc[(df.temp > TEMP_LIMIT) | (df.vib > VIB_LIMIT),
               ["ts", "device_id", "output", "temp", "vib"]].copy()
    both = (a.temp > TEMP_LIMIT) & (a.vib > VIB_LIMIT)
    a["告警原因"] = np.select([both, a.temp > TEMP_LIMIT],
                              ["高温+振动", "温度超限"], default="振动超限")
    a["超限幅度"] = ((a.temp - TEMP_LIMIT).clip(lower=0)
                   + (a.vib - VIB_LIMIT).clip(lower=0)).round(2)   # 严重度: 超阈值累计幅度
    a["处置建议"] = a["告警原因"].map({
        "温度超限": "查冷却泵与散热风道, 复核车间环境温度",
        "振动超限": "复检地脚螺栓/动平衡, 趋势上行则安排换轴承",
        "高温+振动": "立即降载停机点检轴承(高风险), 上报设备主管"})
    return a.sort_values("ts", ascending=False)


def kpis(df):
    """5KPI 统一口径 → dict: 总产量/平均良率/设备在线数/今日告警数/效率对标(+环比)"""
    last = df.ts.max()
    day = df[df.ts >= last.normalize()]                  # "今日"=数据里最后一个自然日
    alarms = alarm_frame(day)
    mid = df.ts.min() + (last - df.ts.min()) / 2         # 前半段 vs 后半段 → metric 的 delta
    prev, cur = df[df.ts < mid], df[df.ts >= mid]
    total = int(df.output.sum())
    return {"rows": int(len(df)),
            "total_output": total,
            "avg_yield": float(df["yield"].mean()),
            "fleet": int(df.device_id.nunique()),
            "online": int(df.loc[df.ts >= last - ONLINE_WINDOW, "device_id"].nunique()),
            "alarm_count": int(len(alarms)),
            "alarms": alarms,
            "achieve": total / TARGET_OUTPUT,
            "out_delta": int(cur.output.sum()) / max(int(prev.output.sum()), 1) - 1,
            "yield_delta": float(cur["yield"].mean()) - float(prev["yield"].mean()),
            "window": f"{df.ts.min():%Y-%m-%d %H:%M} ~ {last:%Y-%m-%d %H:%M}"}


def main():
    st.set_page_config("产线驾驶舱", "📊", layout="wide")
    st.title("🏭 产线KPI实时仪表盘")

    df = load_data()

    # ===== 任务1(学生版第1空) 侧栏筛选: 日期范围(date_input) + 设备多选(multiselect) =====
    # 教师版完成态: value 传元组=区间选择; multiselect 默认全选; 两个掩码用 & 连接
    devs_all = sorted(df.device_id.unique())
    sel = st.sidebar.date_input("日期范围", value=(df.ts.min().date(), df.ts.max().date()),
                                min_value=df.ts.min().date(), max_value=df.ts.max().date())
    d0, d1 = sel if isinstance(sel, tuple) else (sel, sel)     # 兼容单选(学生常见坑)
    devs = st.sidebar.multiselect("设备", devs_all, default=devs_all)
    df = df[df.ts.dt.date.between(d0, d1) & df.device_id.isin(devs)]
    if df.empty:
        st.warning("筛选结果为空: 请放宽日期范围或勾选设备")
        return
    st.sidebar.caption(f"当前视图 {len(df)} 行 · {df.device_id.nunique()} 台 · {d0} → {d1}")

    c1, c2, c3, c4 = st.columns(4)
    # ===== 任务2(学生版第2空) 四个KPI指标卡: 总产量/平均良率/设备在线数/今日告警数 =====
    k = kpis(df)
    c1.metric("总产量", f"{k['total_output']:,}", delta=f"{k['out_delta']:+.1%}")
    c2.metric("平均良率", f"{k['avg_yield']:.2%}", delta=f"{k['yield_delta']:+.2%}",
              delta_color="inverse" if k["yield_delta"] < 0 else "normal")
    c3.metric("设备在线数", f"{k['online']}/{k['fleet']}",
              help=f"最近{ONLINE_WINDOW}内有上报的设备数")
    c4.metric("今日告警数", f"{k['alarm_count']}", delta=f"阈值 温度>{TEMP_LIMIT:.0f}℃/振动>{VIB_LIMIT}",
              delta_color="inverse")

    # ===== 任务3(学生版第3空) 产量趋势: px.line → st.plotly_chart =====
    st.subheader("📈 产量趋势(可悬停看数/框选缩放)")
    trend = df.groupby("ts", as_index=False).output.sum()
    fig_trend = px.line(trend, x="ts", y="output", markers=True,
                        labels={"output": "产量", "ts": "时间"}, title="产线产量趋势(半小时)")
    st.plotly_chart(fig_trend, use_container_width=True)
    fig_dev = px.line(df.groupby(["ts", "device_id"], as_index=False).output.sum(),
                      x="ts", y="output", color="device_id", title="分设备产量趋势")
    st.plotly_chart(fig_dev, use_container_width=True)

    # ===== 任务4(学生版第4空) 良率仪表 + 设备状态红绿灯 + 告警明细表 =====
    st.subheader("🎯 良率仪表 / 设备状态红绿灯 / 告警明细")
    gauge = go.Figure(go.Indicator(
        mode="gauge+number", value=round(k["avg_yield"] * 100, 2),
        number={"suffix": "%", "font": {"size": 40}}, title={"text": "平均良率"},
        gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#1565C0"},
               "threshold": {"line": {"color": "red", "width": 3},
                             "thickness": 0.75, "value": TARGET_YIELD * 100}}))
    st.plotly_chart(gauge, use_container_width=True)

    day = df[df.ts >= df.ts.max().normalize()]
    for dev, g in day.groupby("device_id"):
        n_al = int(((g.temp > TEMP_LIMIT) | (g.vib > VIB_LIMIT)).sum())
        msg = (f"{dev} 峰值振动 {g.vib.max():.2f}mm/s · 峰值温度 {g.temp.max():.1f}℃ "
               f"· 今日告警 {n_al} 条")
        if n_al == 0:
            st.success(msg)                               # 🟢 绿灯: 今日无告警
        elif n_al <= 2:
            st.warning(msg)                               # 🟡 黄灯: 少量超限, 本班次跟踪
        else:
            st.error(msg)                                 # 🔴 红灯: 多点超限, 安排停机点检

    st.dataframe(k["alarms"].head(200), use_container_width=True)
    bar_df = day.groupby(["device_id", "shift"], observed=True)[["output", "ok_qty"]] \
                .sum().reset_index()
    bar_df["shift"] = bar_df["shift"].astype(str)   # ⚠分类列先转str: 否则当日缺席的班次会让px.bar取组KeyError
    fig_bar = px.bar(bar_df, x="device_id", y=["output", "ok_qty"],
                     color="shift", barmode="group", title="今日 设备×班次 产量对比")
    st.plotly_chart(fig_bar, use_container_width=True)

    # ===== 任务5(学生版第5空) 效率对标: 进度条 st.progress(实际/目标) =====
    st.subheader("🏁 效率对标")
    st.progress(min(k["achieve"], 1.0),
                text=f"产量对标 {k['total_output']:,}/{TARGET_OUTPUT:,} = {k['achieve']:.1%}")
    st.progress(min(k["avg_yield"] / TARGET_YIELD, 1.0),
                text=f"良率对标 {k['avg_yield']:.2%}/{TARGET_YIELD:.0%} = "
                     f"{k['avg_yield'] / TARGET_YIELD:.1%}")
    if k["achieve"] < 1:
        need = (TARGET_OUTPUT - k["total_output"]) / max(df.output.mean(), 1)
        st.info(f"缺口 {TARGET_OUTPUT - k['total_output']:,} 件 ≈ 需增加 {need:.0f} 个半小时产能")
    return k


### Starter 4：`build_report()`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [13]:
def _rl_table(df_):
    """reportlab 表格元素: DataFrame → Table(表头每页重复) —— 未装 reportlab 时不会被调用"""
    from reportlab.platypus import Table, TableStyle
    d = df_.astype(str).reset_index(drop=False)
    t = Table([list(d.columns)] + d.values.tolist(), repeatRows=1)
    t.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.4, "black"),
                           ("FONTNAME", (0, 0), (-1, -1), "ZH"),
                           ("FONTSIZE", (0, 0), (-1, -1), 7.5),
                           ("BACKGROUND", (0, 0), (-1, 0), "#dce6f1"),
                           ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    return t


def to_pdf(k, shift_tab, dev_tab, anom, pdf_path="daily_report.pdf"):
    """工单P3.2 PDF 交付: 复用02包W10字体方案(reportlab 默认字体无中文 → 注册 fonts/ZH_Font.ttf)。
    本机未装 reportlab → 打印降级说明, HTML 仍是完整四段式交付物。"""
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont
        from reportlab.platypus import Paragraph, SimpleDocTemplate
    except ImportError:
        print("  · reportlab 未安装 → 本次只交付 HTML(装好 reportlab 后再执行 build_report() "
              "即自动出 PDF: pip install reportlab)")
        return None
    try:
        pdfmetrics.registerFont(TTFont("ZH", FONT_PATH))
        ss = getSampleStyleSheet()
        for name in ("Title", "Normal", "Heading2"):
            ss[name].fontName = "ZH"
        story = [Paragraph("每日生产分析报告", ss["Title"]),
                 Paragraph(f"数据区间 {k['window']} · 样本 {k['rows']:,} 行", ss["Normal"]),
                 Paragraph("1. 生产概览", ss["Heading2"]),
                 Paragraph(f"总产量 {k['total_output']:,} 件 · 平均良率 {k['avg_yield']:.2%} · "
                           f"设备在线 {k['online']}/{k['fleet']} · 今日告警 {k['alarm_count']} 条",
                           ss["Normal"]),
                 Paragraph("2. 详细分析(班次/设备对比)", ss["Heading2"]), _rl_table(shift_tab),
                 _rl_table(dev_tab),
                 Paragraph("3. 异常识别(温度>90℃或振动>4.5mm/s)", ss["Heading2"]),
                 _rl_table(anom.head(15)),
                 Paragraph("4. 数据附表(末50行)", ss["Heading2"])]
        SimpleDocTemplate(pdf_path, pagesize=A4).build(story)
        print(f"  · PDF 已生成 {pdf_path}")
        return pdf_path
    except Exception as e:                 # 字体缺失/reportlab版本差异 → 不影响HTML交付
        print(f"  · PDF 生成失败({e.__class__.__name__}: {e}) → HTML 交付不受影响")
        return None


def build_report(path=REPORT_PATH):
    """P3.2 每日日报(四段式: 概览/分析/异常/附表) — python W05_dashboard.py --report 触发
    教师版完成态: HTML 用 f-string 拼装(工单里的 Jinja2 是同构写法, 见《多解法手册》), 并尽力转PDF"""
    t0 = time.time()
    df = load_data()
    k = kpis(df)
    hit = (df.temp > TEMP_LIMIT) | (df.vib > VIB_LIMIT)     # 全量异常掩码(口径同 alarm_frame)

    # ---- 段2 数据: 班次对比表 + 设备对比表(工单"详细分析") ----
    shift_tab = df.groupby("shift", observed=True)[["output", "ok_qty", "bad_qty"]].sum()
    shift_tab["良率"] = (shift_tab.ok_qty / shift_tab.output).round(4)
    shift_tab["告警条数"] = (df.loc[hit].groupby("shift", observed=True).size()
                          .reindex(shift_tab.index, fill_value=0))
    dev_tab = df.groupby("device_id")[["output", "ok_qty", "bad_qty"]].sum()
    dev_tab["良率"] = (dev_tab.ok_qty / dev_tab.output).round(4)
    dev_tab["平均温度℃"] = df.groupby("device_id").temp.mean().round(1)
    dev_tab["峰值振动"] = df.groupby("device_id").vib.max().round(2)
    dev_tab["全期告警"] = (df.loc[hit].groupby("device_id").size()
                        .reindex(dev_tab.index, fill_value=0))
    dev_tab["今日告警数"] = (k["alarms"].device_id.value_counts()
                        .reindex(dev_tab.index, fill_value=0))

    # ---- 段3 数据: 异常识别 + 建议(全量口径; 报表展示前30条) ----
    anom = alarm_frame(df)
    reason_stat = anom["告警原因"].value_counts()
    worst = anom.nlargest(5, "超限幅度")          # 按严重度(超阈值累计幅度)取前5
    v_yield = "达标" if k["avg_yield"] >= TARGET_YIELD else "未达标(低于目标线)"
    v_out = "已达成" if k["achieve"] >= 1 else f"未达成(缺口 {TARGET_OUTPUT - k['total_output']:,} 件)"
    tail50 = df.tail(50)[["ts", "device_id", "output", "ok_qty", "yield", "temp", "vib", "shift"]]

    css = ("<style>body{font-family:'Microsoft YaHei',sans-serif;margin:34px;color:#222}"
           "h1{border-bottom:3px solid #1565C0;padding-bottom:6px}"
           "h2{color:#1565C0;margin-top:26px}table{border-collapse:collapse;font-size:12.5px}"
           "td,th{border:1px solid #b9c6d6;padding:3px 8px}"
           "tr:nth-child(even){background:#f4f7fb}"
           ".kpi{display:flex;gap:18px;flex-wrap:wrap}.card{background:#f4f7fb;"
           "border-left:4px solid #1565C0;padding:10px 16px;min-width:160px}"
           ".bad{color:#b71c1c;font-weight:700}.muted{color:#666;font-size:12px}</style>")

    html = f"""<!DOCTYPE html><html lang="zh-CN"><head><meta charset="utf-8">
<title>每日生产分析报告</title>{css}</head>
<body style="font-family:sans-serif">
<h1>每日生产分析报告 · {datetime.date.today()}</h1>
<p class="muted">数据区间 {k['window']} · 样本 {k['rows']:,} 行 ·
生成耗时 {time.time() - t0:.2f}s（工单验收 &lt; 2 分钟）</p>

<h2>1. 生产概览</h2>
<div class="kpi">
  <div class="card">总产量<br><b>{k['total_output']:,}</b> 件</div>
  <div class="card">平均良率<br><b>{k['avg_yield']:.2%}</b>（目标 {TARGET_YIELD:.0%}）</div>
  <div class="card">设备在线<br><b>{k['online']}/{k['fleet']}</b> 台</div>
  <div class="card">今日告警<br><b class="bad">{k['alarm_count']}</b> 条</div>
  <div class="card">产量对标<br><b>{k['achieve']:.1%}</b></div>
</div>
<p>结论：良率 <b>{v_yield}</b>；产量目标 <b>{v_out}</b>；
今日 {k['alarm_count']} 条告警已在第3节逐条给出处置建议。</p>

<h2>2. 详细分析</h2>
<h3>2.1 班次对比（产量/合格/缺陷/良率/告警）</h3>{shift_tab.to_html(border=0)}
<h3>2.2 设备对比（良率与峰值指标）</h3>{dev_tab.to_html(border=0)}
<p>读数要点：良率最低班次 <b>{shift_tab["良率"].idxmin()}</b>（{shift_tab["良率"].min():.2%}），
告警最多设备 <b>{dev_tab["今日告警数"].idxmax()}</b>（{int(dev_tab["今日告警数"].max())} 条）——
优先安排该设备的点检与备件。</p>

<h2>3. 异常识别</h2>
<p>判定规则：温度 &gt; {TEMP_LIMIT:.0f}℃ 或 振动 &gt; {VIB_LIMIT} mm/s；
全量命中 <b class="bad">{len(anom)}</b> 条，占比 {len(anom) / max(len(df), 1):.1%}。
原因分布：{dict(reason_stat)}。</p>
<h3>3.1 最严重 5 条（按超限幅度排序）与建议</h3>{worst.to_html(index=False, border=0)}
<h3>3.2 异常明细（前 30 条）</h3>{anom.head(30).to_html(index=False, border=0)}
<p>建议模板：高温→查冷却泵/散热风道；高振动→复检地脚螺栓与动平衡；
两者同时超限→降载停机点检轴承并上报设备主管。</p>

<h2>4. 数据附表</h2>
<p class="muted">末 50 行原始记录（工单要求 df.tail(50).to_html()）</p>
{tail50.to_html(index=False, border=0)}
</body></html>"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"日报已生成 {path}: 概览/分析/异常/附表 四段完整, {len(html):,} 字符, "
          f"异常 {len(anom)} 条, 耗时 {time.time() - t0:.2f}s")
    to_pdf(k, shift_tab, dev_tab, anom)                 # 工单要求 HTML+PDF 双交付
    return path


### Starter 5：`if __name__ == "__main__"`
（教师版：下方为完成态可运行代码；学生版此处为待填空位）


In [15]:
if __name__ == "__main__":
    # 教师版完成态: 与 .py 脚本入口一致 —— 命令行带 --report 出日报, 否则起仪表盘
    import sys
    if "--report" in sys.argv:
        build_report()
    else:
        main()                # 课堂: streamlit run W05_dashboard.py


2026-10-28 03:46:28.444 
  command:

    streamlit run /tmp/qoder_nb/capture.py [ARGUMENTS]
2026-10-28 03:46:28.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.444 No runtime found, using MemoryCacheStorageManager
2026-10-28 03:46:28.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 03:46:28.466 Thread 'MainThread': missing ScriptRunContext! This wa

### 教师版演示：headless 跑通 `build_report()`（等价于 `--report` 分支）
notebook 里 `sys.argv` 没有 `--report`，所以这里显式调用一次并**断言四段式完整落盘**；
`kpis()` 与看板取同一份缓存数据，故日报数字与上方指标卡完全一致（可用输出的数字当面核对）。


In [17]:
# 教师版完成态: 直接演示"生成日报" —— 落盘 + 四段式完整性断言(验收: 概览/分析/异常/附表)
out_path = build_report()
assert os.path.exists(out_path), "日报未落盘"
html = open(out_path, encoding="utf-8").read()
for sec in ("1. 生产概览", "2. 详细分析", "3. 异常识别", "4. 数据附表"):
    assert sec in html, f"四段式缺少段落: {sec}"
assert "<!--" not in html, "残留未替换的占位注释"
assert html.count("<table") >= 4, f"表格数不足: {html.count('<table')}"
print("日报四段式校验通过 ✓  字节数:", len(html.encode("utf-8")))

k_dash = kpis(load_data())                  # 看板与日报必须同一口径(交叉核对数字)
for token in (f"{k_dash['total_output']:,}", f"{k_dash['avg_yield']:.2%}",
              f"{k_dash['alarm_count']}"):
    assert token in html, f"日报未包含看板口径 {token}"
print("看板与日报口径一致 ✓  总产量", f"{k_dash['total_output']:,}",
      "平均良率", f"{k_dash['avg_yield']:.2%}", "今日告警", k_dash["alarm_count"])


日报已生成 daily_report.html: 概览/分析/异常/附表 四段完整, 22,824 字符, 异常 755 条, 耗时 0.04s
  · PDF 已生成 daily_report.pdf
日报四段式校验通过 ✓  字节数: 25349
看板与日报口径一致 ✓  总产量 100,414 平均良率 92.48% 今日告警 11


## 四、提交要求与避坑
- 提交: `学号_姓名_P3.py` 或本notebook(含输出)
- ⚠ Plotly图记得st.plotly_chart(fig)渲染——只有 plotly 图对象能进，matplotlib 对象不行
- ⚠ 时间筛选用 st.date_input 的区间值（value 传元组），单选会退化成标量 → 先 `isinstance(sel, tuple)` 兼容
- ⚠ 缓存键=数据文件mtime/函数入参，数据更新后要 `st.cache_data.clear()` 或设 `ttl`
- ⚠ `df["yield"]`：`yield` 是 Python 关键字，`df.yield` 直接语法错误
- ⚠ `st.progress()` 只吃 0~1，超目标要 `min(比率, 1.0)`，否则 ValueError


---

## 教师参考答案（与 `教师参考/solutions.py` · `p33()` 同源，本 notebook 内已可运行）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版 5 处待填空位 → 下方参考实现的对应关系：

| 学生版待填空位 | 所在函数 | 教师完成态所在 cell | 关键实现要点 |
|---|---|---|---|
| 任务1 侧栏筛选（日期区间+设备多选） | `main()` | Starter 3（cell 10） | `date_input` 传元组=区间；`multiselect` 默认全选；`df[m1 & m2]` 掩码切片 |
| 任务2 四个KPI指标卡 | `main()` | Starter 3（cell 10） | `output.sum()` / `df["yield"].mean()` / 2h内有上报的 `nunique()` / `temp>90 \| vib>4.5` 计数 |
| 任务3 产量趋势折线 | `main()` | Starter 3（cell 10） | `groupby("ts")` 汇总 → `px.line(...)` → `st.plotly_chart(fig)` |
| 任务4 良率仪表+状态灯+告警表 | `main()` | Starter 3（cell 10） | `go.Indicator` gauge(阈值95%)；按告警条数 `success/warning/error`；`st.dataframe` |
| 任务5 效率对标进度条 | `main()` | Starter 3（cell 10） | `st.progress(min(实际/目标, 1.0), text=…)`，产量与良率各一条 |
| 数据层（缓存+可选 clean.csv） | `load_data()` | Starter 2（cell 7） | `@st.cache_data` 重跑不重算；补 `shift/day/bad_qty` 派生列 |
| 日报四段式（班次/设备对比+异常+附表+落盘） | `build_report()` | Starter 4（cell 12） | `to_html()` 三段表 + `alarm_frame()` 建议 + `tail(50)` 附表 + 尽力转 PDF |
| 离线降级（本机无 streamlit） | 顶部导入块 | Starter 1（cell 5） | `try/except ImportError` + `_HeadlessContainer` 打印渲染意图 |

> 下方 `p33()` 为**可直接运行**的参考答案（含 KPI 与日报完整性断言），与上方完成态 cell 互为校验。
> 注：表内 cell 编号为本教师版 notebook 的当前序号，讲解时按 Starter 小标题定位更稳。


In [20]:
def p33():
    """教师参考答案(P3): 五KPI + 四段式日报的最小可运行实现 + 断言
    —— 与上方完成态互为校验；合成数据、纯离线、结果可复现(装了 streamlit 也照跑)"""
    df = load_data()
    k = kpis(df)

    # 1) KPI 口径逐条与手算对齐(课堂强调: 看板数字必须能对得上账)
    assert k["total_output"] == int(df.output.sum())
    assert k["rows"] == len(df) == 2000
    assert abs(k["avg_yield"] - df["yield"].mean()) < 1e-12
    weighted = df.ok_qty.sum() / df.output.sum()        # 加权良率(件口径)
    assert abs(k["avg_yield"] - weighted) < 0.02        # 逐点均值 vs 加权: 差<2个点, 口径要写清
    last = df.ts.max()
    assert 0 <= k["online"] <= k["fleet"] <= 4
    assert k["online"] == int(df[df.ts >= last - ONLINE_WINDOW].device_id.nunique())
    manual_alarm = int(((df[df.ts >= last.normalize()].temp > TEMP_LIMIT) |
                        (df[df.ts >= last.normalize()].vib > VIB_LIMIT)).sum())
    assert k["alarm_count"] == manual_alarm > 0         # 今日告警数=阈值命中条数
    assert set(k["alarms"]["告警原因"]) <= {"温度超限", "振动超限", "高温+振动"}
    assert k["alarms"]["处置建议"].notna().all()        # 每条异常都给了建议

    # 2) 三张 Plotly 图确实建出来了(离线也建, 装了 streamlit 就 st.plotly_chart 渲染)
    fig_line = px.line(df.groupby("ts", as_index=False).output.sum(), x="ts", y="output")
    fig_bar = px.bar(df.groupby("device_id")[["output"]].sum().reset_index(),
                     x="device_id", y="output")
    fig_gauge = go.Figure(go.Indicator(mode="gauge+number", value=k["avg_yield"] * 100,
                                      gauge={"axis": {"range": [0, 100]},
                                             "threshold": {"value": TARGET_YIELD * 100}}))
    assert isinstance(fig_line, go.Figure) and isinstance(fig_bar, go.Figure)
    assert str(fig_line.data[0].type).startswith("scatter") and fig_bar.data[0].type == "bar"
    assert fig_gauge.data[0]["gauge"]["threshold"]["value"] == 95
    print(f"图对象就绪: 折线{len(fig_line.data)}条 / 柱状{len(fig_bar.data)}条 / "
          f"仪表读数{fig_gauge.data[0].value:.2f}%")

    # 3) 日报四段式落盘 + 完整性
    path = build_report()
    html = open(path, encoding="utf-8").read()
    for sec in ("1. 生产概览", "2. 详细分析", "3. 异常识别", "4. 数据附表"):
        assert sec in html, f"日报缺段: {sec}"
    assert "温度" in html and "振动" in html and html.count("<table") >= 4
    assert f"{k['total_output']:,}" in html and "<!--" not in html
    assert k["alarms"]["处置建议"].str.len().min() > 5, "异常记录必须逐条给建议"

    # 4) 缓存语义: @st.cache_data 二次调用不重算(真 streamlit 返回副本 → 用 equals 比较)
    assert load_data().equals(df)
    print(f"P3 参考: 总产量 {k['total_output']:,} | 平均良率 {k['avg_yield']:.2%} | "
          f"在线 {k['online']}/{k['fleet']} | 今日告警 {k['alarm_count']} | "
          f"对标 {k['achieve']:.1%} | 全量异常 {len(alarm_frame(df))} 条 | 日报 {path}")
    print(f"环境: streamlit={'已安装(真控件渲染)' if HAVE_ST else '未安装(headless降级打印)'}")
    print("P3 参考答案断言全部通过 ✓  (D层=5KPI+筛选 / B层=联动+Plotly交互 / A层=钻取扩展)")
    return k


p33()


图对象就绪: 折线1条 / 柱状1条 / 仪表读数92.48%
日报已生成 daily_report.html: 概览/分析/异常/附表 四段完整, 22,824 字符, 异常 755 条, 耗时 0.04s
  · PDF 已生成 daily_report.pdf
P3 参考: 总产量 100,414 | 平均良率 92.48% | 在线 3/4 | 今日告警 11 | 对标 95.6% | 全量异常 755 条 | 日报 daily_report.html
环境: streamlit=已安装(真控件渲染)
P3 参考答案断言全部通过 ✓  (D层=5KPI+筛选 / B层=联动+Plotly交互 / A层=钻取扩展)


## 运行与验证

**本教师版已在 `/usr/local/bin/python3.11` 下逐 cell 跑通**（无 streamlit，走 headless 降级）。

| cell | 内容 | 执行结果 |
|---|---|---|
| 2 | 环境准备：`os.chdir("../")` 切到课程包根 | 工作目录=课程包根，`clean.csv/fonts/` 均可见 |
| 5 | 导入 + 离线降级 shim + KPI 阈值常量 | `HAVE_ST=False`，`st` 为 headless 容器 |
| 7 | `load_data()`（`@st.cache_data`） | 2000 行 × 9 列（含 shift/day/bad_qty） |
| 10 | `alarm_frame()/kpis()/main()` 完成态 | 5 KPI + 4 张 Plotly 图 + 状态灯 + 告警表 |
| 12 | `build_report()/to_pdf()` 完成态 | 四段式 `daily_report.html` 落盘；reportlab 缺失→打印降级说明 |
| 14 | 脚本入口守卫（`--report` 分支） | notebook 内 `sys.argv` 无 `--report` → 执行 `main()` |
| 16 | 新增演示 cell | `build_report()` + 四段式断言 + 看板/日报口径交叉核对 |
| 19 | `p33()` 参考答案 | KPI/图对象/日报/缓存共 15 项断言全绿 |

**重跑方式**
- Jupyter：`Kernel → Restart` 后 `Run All`（约 3 秒；`_executed.ipynb` 为留档版，含全部输出）。
- 命令行等价：`/usr/local/bin/python3.11 /tmp/qoder_nb/capture.py W05_..._教师版.ipynb W05_..._教师版_executed.ipynb Jupyter实训`
- 装了 streamlit 的机器：把 cell 5~12 原样存成 `W05_dashboard.py`，`streamlit run W05_dashboard.py`
  即得真控件仪表盘（代码一行不改）；`python W05_dashboard.py --report` 出日报 + PDF。
- 日报输出：课程包根目录 `daily_report.html`（每次运行覆盖，可用浏览器打开自查四段）。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。
